<a href="https://colab.research.google.com/github/makaog5-sudo/CRM-SP500/blob/main/CRM_stock_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install yfinance pandas numpy


In [4]:
import pandas as pd, numpy as np, yfinance as yf, sklearn, scipy
from datetime import datetime, timezone

# --- Record environment for reproducibility ---
print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)

# --- Task 1.1: Extract S&P 500 tickers from Wikipedia ---
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# User-Agent avoids HTTP 403 blocking from Wikipedia
tables = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500 = next(
    table for table in tables
    if "Symbol" in table.columns and "Security" in table.columns
)

# Record the retrieval date
retrieval_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Convert ticker symbols for Yahoo Finance (e.g. BRK.B -> BRK-B)
sp500["Yahoo_Ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)

# Save the list with its retrieval date
sp500["Retrieval_Date_UTC"] = retrieval_date
sp500.to_csv("sp500_tickers.csv", index=False)

tickers = sp500["Yahoo_Ticker"].tolist()

print("Retrieval date:", retrieval_date)
print("Total tickers:", len(tickers))
print("First 10 tickers:", tickers[:10])

sp500[["Symbol", "Yahoo_Ticker", "Security", "GICS Sector"]].head()

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66 | sklearn 1.6.1 | scipy 1.16.3
Retrieval date: 2026-11-17
Total tickers: 503
First 10 tickers: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


,Symbol,Yahoo_Ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology


In [6]:

# Download all S&P 500 stocks and the benchmark
all_tickers = tickers + ["^GSPC"]

stock_data = yf.download(
    all_tickers,
    start="2021-01-01",
    end="2025-01-01",
    auto_adjust=False,
    group_by="column",
    threads=True,
    progress=True
)

print("Dataset shape:", stock_data.shape)
print("Number of trading days:", len(stock_data))

stock_data.head()


[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['FDXF', 'SNDK', 'VYLR', 'Q', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Dataset shape: (1005, 3024)
Number of trading days: 1005


Price        Adj Close                                                \
Ticker               A        AAPL       ABBV        ABNB        ABT   
Date                                                                   
2021-01-04  114.009117  125.537125  84.219139  139.149994  97.269707   
2021-01-05  114.941246  127.089241  85.090012  148.300003  98.473183   
2021-01-06  118.093216  122.811218  84.354965  142.770004  98.268166   
2021-01-07  121.235588  127.001915  85.257790  151.270004  99.222046   
2021-01-08  122.100471  128.098129  85.705208  149.770004  99.498405   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2021-01-04  33.186359  232.029343  485.339996  132.622665  42.507668  ...   
2021-01-05  33.319485  233.350296  485.690002  134.993149  43.334469  ...   
2021-01-06  34.783871  235.901642  466.309998  135.601715  45.090340  ...   
2021-01-07  34.460564  238.127319  477.739990  141.332764  44.689732  ...   
2021-01-08  34.650742  238.995850  485.100006  142.359039  44.809067  ...   

Price        Volume                                                          \
Ticker         WYNN      XEL       XOM      XYL       XYZ      YUM      ZBH   
Date                                                                          
2021-01-04  3596800  3197100  27764700   814700   9838400  2058200  1265870   
2021-01-05  1814600  1976800  44035100  1005200   9699400  1475300   998894   
2021-01-06  2433700  2667000  36484800  1015300  12003500  1352700  1424799   
2021-01-07  3846700  2630400  29528100  1539300  10214200  2222900  1237648   
2021-01-08  2812400  2221500  28628200  1632200   9102100  1252200  1259072   

Price                                    
Ticker        ZBRA      ZTS       ^GSPC  
Date                                     
2021-01-04  432600  1805900  5015000000  
2021-01-05  374700  1358600  4591020000  
2021-01-06  419500  2139800  6064110000  
2021-01-07  379900  1710700  5099160000  
2021-01-08  348500  1475900  4773040000  

[5 rows x 3024 columns]

In [10]:
raw = stock_data
stock_data.to_pickle("raw_prices_2021_2024.pkl.gz")

print("Stock data saved successfully!")


Stock data saved successfully!
